# Causally-informed sparse risk scoring

A method for interpretable integer-coefficient risk scoring with a soft, threshold-free *causal* prior on feature selection. The TB-DLM dataset is one application; the method composes with any causal-evidence vector over features.

**Notation.** We follow Liu et al. (2022, FasterRisk): dataset $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $\mathbf{x}_i \in \mathbb{R}^p$ and $y_i \in \{-1, +1\}$; coefficient vector $\mathbf{w} \in \mathbb{Z}^p$ with intercept $w_0$; sparsity $k$; coefficient bound $C$; multiplier $m > 0$; integer-rounded solution $\mathbf{w}^+$. Sample index $i \in [1, n]$, feature index $j \in [1, p]$.

## 1. Problem setup

Inputs to the FasterRisk classifier stage:

- Features $X \in \mathbb{R}^{n \times p}$, mixed binary and continuous.
- Binary target $y \in \{-1, +1\}^n$ (required by FasterRisk's logistic loss).
- Causal-prior vector $\mathbf{q} \in [0, 1]^p$, where $q_j$ encodes evidence that feature $j$ is a *cause* of the underlying signal (not merely a correlate).

The vector $\mathbf{q}$ is produced by a separate procedure (Section 3) whose target need not be binary: it can be continuous, ordinal, or multi-class as appropriate for the prior source. The TB application uses CMM on continuous $\log_2(\mathrm{MIC})$ at the prior-estimation stage; binarization happens only at the FasterRisk stage and is a constraint imposed by FasterRisk's design, not by the problem.

Output: integer-coefficient sparse logistic classifier whose support is biased toward features with high $q_j$, with bias strength controlled by a single continuous parameter $\mu \geq 0$.

## 2. Method

### 2.1 Modified objective

Standard FasterRisk (Liu et al. 2022, Eq. 1):

$$
\min_{\mathbf{w}, w_0} \; L(\mathbf{w}, w_0, \mathcal{D}) \quad \text{s.t.} \quad \|\mathbf{w}\|_0 \leq k, \; \mathbf{w} \in \mathbb{Z}^p, \; w_j \in [-C, C], \; w_0 \in \mathbb{Z},
$$

where

$$
L(\mathbf{w}, w_0, \mathcal{D}) = \sum_{i=1}^n \log\!\left(1 + \exp\!\left(-y_i (\mathbf{x}_i^\top \mathbf{w} + w_0)\right)\right).
$$

Modified objective with causal-prior bonus:

$$
\min_{\mathbf{w}, w_0} \; L(\mathbf{w}, w_0, \mathcal{D}) \;-\; \mu \sum_{j=1}^p q_j \cdot \mathbb{1}[w_j \neq 0]
$$

subject to the same constraints. Parameters: $\mu \geq 0$ (prior strength), $k$ (sparsity), $C$ (coefficient bound). The construction is solver-agnostic: the same modified objective is targeted by both FasterRisk (beam-search heuristic) and RiskSlim (exact MIP); see §6.

### 2.2 MAP derivation

Place a Bernoulli inclusion prior on support indicators $z_j = \mathbb{1}[w_j \neq 0]$ with $z_j \sim \text{Bernoulli}(\pi_j)$ independently across $j$, and a uniform conditional prior on $w_j \mid z_j = 1$ over $\{-C, \ldots, -1, 1, \ldots, C\}$. This is the **discrete spike-and-slab** decomposition (Mitchell & Beauchamp 1988, George & McCulloch 1993) adapted to integer coefficients. The log-prior reduces to

$$
\log p(\mathbf{w}) = \sum_{j=1}^p z_j \, \mathrm{logit}(\pi_j) + \text{const}.
$$

Setting $\pi_j = \sigma(\mu \cdot q_j)$:

$$
\log p(\mathbf{w}) = \mu \sum_{j=1}^p q_j \, \mathbb{1}[w_j \neq 0] + \text{const}.
$$

MAP estimation minimizes $L(\mathbf{w}, w_0, \mathcal{D}) - \log p(\mathbf{w})$, which is the modified objective up to a constant. The sigmoid link is the *unique* choice making the log-prior linear in $q_j$ with slope $\mu$; the linear-in-$\mathbf{q}$ form is not an ad-hoc functional pick. Note: $q_j = 0$ gives $\pi_j = 1/2$ (uniform inclusion at zero evidence); the prior treats $q$ as evidence on a positive scale, not as a probability centered at $1/2$. Independence across $j$ is the load-bearing assumption — it gives per-feature decomposability and matches the per-feature nature of the causal evidence $q_j$; collinearity-induced redundancy is not captured by the prior and is mitigated only by the hard sparsity cap (§8).

### 2.3 Limits

$\mu = 0$: $\pi_j = 1/2$, uniform inclusion prior, vanilla FasterRisk recovered.

$\mu \to \infty$: $\pi_j \to 1$ for any $q_j > 0$; the $k$-constrained optimum collapses to the support maximizing $\sum_{j \in S} q_j$ (top-$k$ by $\mathbf{q}$, modulo beam-search heuristics; see implementation cell §5.1).

### 2.4 Structural properties

**Linear separability across features.** The bonus decomposes as $\sum_j q_j \cdot \mathbb{1}[w_j \neq 0]$. Per-feature marginal cost is computable without recomputing global support quantities. FasterRisk's SparseBeamLR expansion (Liu et al. Alg. 2, with ExpandSuppBy1 as Alg. 4) and CollectSparseDiversePool swap (Alg. 5) remain per-feature decomposable.

**Magnitude invariance under integer rounding** (conditional on support preservation). The bonus depends only on $\mathbb{1}[w_j \neq 0]$, not on $|w_j|$. FasterRisk's AuxiliaryLossRounding (Alg. 6) preserves support whenever the StarRaySearch multiplier $m$ is large enough that no $|m \cdot w_j|$ for $j \in \mathrm{supp}(\mathbf{w})$ rounds to zero. Under this condition, Theorem 3.1's multiplicative bound transfers to the modified objective:

$$
L(\mathbf{w}^+, w_0^+, \mathcal{D}) - L(\mathbf{w}, w_0, \mathcal{D}) \;\leq\; \sqrt{\,n \sum_{i=1}^n \sum_{j=1}^p \ell_i^2 \, x_{ij}^2 \, u_j (1 - u_j)\,},
$$

with $u_j = w_j - \lfloor w_j \rfloor$ and $\ell_i = 1 / (1 + \exp(y_i \mathbf{x}_i^\top \boldsymbol{\gamma}_i))$ as in Liu et al. The bonus term in the modified objective is identically zero across rounding because support is invariant. Support preservation can fail at extreme $\mu$ when the bonus forces low-$|w_j|$ features into the support (§8); empirically mitigated by FasterRisk's multiplier search.

**Scale.** $\mu$ has no data-invariant scale: $L(\mathbf{w}, w_0, \mathcal{D})$ grows with $n$ while $\mathbf{q} \in [0, 1]^p$ is unit-free. Practical convention: report $\mu$ in units of $\mathrm{median}_j \,|\nabla_j L|$ evaluated at $\mathbf{w} = \mathbf{0}$, computed once per dataset.

### 2.5 Implementation

Local modifications to FasterRisk's `sparseBeamSearch.py` (SparseBeamLR, Alg. 2) and `sparseDiversePool.py` (CollectSparseDiversePool, Alg. 5). Pure Python; no asymptotic complexity change. Numerically equivalent to vanilla FasterRisk at $\mu = 0$ or $\mathbf{q} = \mathbf{0}$. FasterRisk's implementation also includes a small $L_2$ ridge $\lambda_2 \|\mathbf{w}\|^2$ as numerical regularization (default $\lambda_2 = 10^{-8}$, not in Liu et al. Eq. 1); we preserve this exactly. For the RiskSlim instantiation (§6) the penalty is added as a linear cost coefficient on the inclusion indicators in the MIP objective — a one-line edit. Full change-point inventory in the implementation cell.

## 3. The causal-prior constraint

**Requirement.** $\mathbf{q}$ must come from a procedure that performs conditional-independence reasoning to remove confounding-driven associations. Predictive-only signals (bootstrap stability of LASSO/RF/XGBoost, marginal mutual information, model-based feature importance) are **not admissible sources**. They measure the same quantity $L(\mathbf{w}, w_0, \mathcal{D})$ already optimizes; treating them as a prior puts the loss against itself and adds no information.

Causal $\mathbf{q}$ adds information the loss cannot recover: which features are *causes* of the target versus features that merely covary with causes through confounding paths. This distinction is the entire point of the contribution; without it the method degenerates to a sparse-classifier-with-side-information.

The prior-estimation procedure operates on whatever target type it accepts (continuous, binary, multi-class). Any binarization happens later, only at the FasterRisk classifier stage.

### 3.1 Admissible sources

1. **PC algorithm** (Spirtes et al.). Causal sufficiency assumed; fast; works on mixed types via appropriate conditional-independence tests. The default cheap option, used as the prior source on the §6.2 standard benchmarks.
2. **GES** (Chickering). Score-based DAG learner; causal sufficiency assumed.
3. **IAMB / HITON-MB.** Markov-blanket-targeted; appropriate when only $\mathrm{MB}(t)$ is required.
4. **Invariant Causal Prediction** (Peters et al., 2016). Pooled-environment regression with invariance-based selection; produces $p$-value-derived $q_j$ in multi-environment settings.
5. **Knowledge graphs with directional causal edges.** Posterior causal-edge probability from curated bases (e.g., gene regulatory networks with established directionality).
6. **Expert causal elicitation.** Per-feature inclusion probabilities from a domain expert, explicitly conditioned on causal status rather than predictive utility.
7. **CMM-logistic** (Mameche et al.) with FLXMRglm binomial extension. Admits latent confounders via per-node mixture components. Handles continuous and binary node types natively. The TB-specific choice (Section 7), defensible only on TB-shaped data (binary low-prevalence features with a continuous target and unobserved confounders).

All sources are used via subsample stability selection ($B$ runs, frequency $q_j = \mathrm{freq}(j \to t)$). Comparing source quality across these is an empirical question (Section 6). The MAP construction holds regardless of which source produces $\mathbf{q}$.

## 4. Inherited and missing guarantees

**Inherited.**

- Integer-rounding bound (Liu et al. 2022, Thm. 3.1) on $L$ transfers exactly to $L_{\text{mod}} = L - \mu \sum_j q_j \mathbb{1}[w_j \neq 0]$ under support preservation.
- For graph-based sources at subsample fraction $\leq 1/2$, Meinshausen-Bühlmann FWER bounds apply to $\mathbf{q}$. The 80% subsampling fraction used in the TB application trades this guarantee for finite-sample power.

**Not inherited.**

- No end-to-end consistency for the two-stage pipeline (prior estimation followed by MAP selection).
- No closed-form $\mu$ scaling against $n$.

## 5. Theoretical targets

Ordered by feasibility within the implementation window.

1. **Robustness to prior perturbation.** For $\mu$ above a separation threshold $\mu_0$ (depending on the loss-curvature gap between the $k$-th and $(k{+}1)$-th best supports), a change of $\epsilon$ in $\|\mathbf{q}\|_\infty$ from finite-$B$ noise changes the MAP-selected support by $O(\epsilon / \mu)$. Mechanical perturbation analysis.
2. **Rashomon pool as posterior region.** Characterize the FasterRisk Rashomon pool $\mathcal{P}$ (output of CollectSparseDiversePool) as an approximation to the posterior mode region under the Bernoulli inclusion prior. Quantify the approximation gap induced by gap-tolerance on the raw loss.
3. **(Advisor-dependent.) Consistency.** Under faithfulness, prior-source identifiability, and $\mathbf{q} \to \mathbb{1}[j \in \mathrm{MB}(t)]$ as $B \to \infty$, MAP recovers the true MB of the prior-source target $t$ as $n, B \to \infty$.

## 6. Evaluation framework

### 6.1 Synthetic validation (primary)

Planted sparse-logistic ground truth with a known causal DAG; $X$ generated with controlled prevalence and pairwise correlation; $\mathbf{q}$ constructed with controllable signal-to-noise relative to the true causal parents. Sweep: noise level on $\mathbf{q}$, prior-target alignment, $\mu$, $n / p$, sparsity $k$.

Metrics: causal support recovery, AUC, Brier, runtime.

Adversarial regimes: $\mathbf{q}$ anti-aligned with truth, faithfulness violations, latent confounders the prior source cannot absorb, mixed causal/predictive priors.

Dual-solver: both FasterRisk and RiskSlim run wherever $p$ is in the MIP-tractable range ($p \leq 30$ comfortable, $p \leq 50$ feasible); FasterRisk-only above. Cross-solver agreement at fixed $(\mu, k)$ is part of the validation — if the two solvers disagree at $\mu = 0$, FasterRisk's beam search is loose; if they disagree the same way at $\mu > 0$, the modification is fine and we have an honest read on beam-search error.

### 6.2 Standard benchmarks

Subset of the FasterRisk/RiskSLIM evaluation suite (mammo, COMPAS, FICO, bankruptcy, mushroom). Prior source: PC algorithm with subsampling stability ($B = 100$, fraction $1/2$ to preserve M-B FWER control). Selected as the cheapest admissible source under causal sufficiency, plausible on these curated tabular benchmarks. These benchmarks have natively binary targets, so the two-stage architecture collapses to one stage. Dual-solver $2 \times 2$ {vanilla, causal} $\times$ {FasterRisk, RiskSlim} wherever $p \leq 30$ (mammo, COMPAS, bankruptcy, FICO at moderate granularity, mushroom borderline); FasterRisk-only above.

### 6.3 Domain case study: TB-DLM

Three sub-experiments, distinguished by feature-set dimension. Same data ($n = 164$); different choices of feature pre-filtering. The prevalence filter $[5\%, 98\%]$ is a data-quality gate (drops near-constant features); it applies to all arms identically and is not a feature-selection method.

**Experiment 3.1 — Raw mutation set ($p = 211$, FasterRisk only).** Both arms operate on the full mutation matrix. The causal-prior vector $\mathbf{q} \in [0, 1]^{211}$ has nonzero entries only on the 17 prevalence-feasible features that CMM could evaluate; $q_j = 0$ for the remaining 194 features encodes "no causal evidence available," which at $\mu = 0$ contributes uniformly and induces no penalty asymmetry vs vanilla. RiskSlim does not run here ($p$ is well outside MIP-tractable range at this $n$). The interest is the $p \gg n$ regime where vanilla overfits and the causal prior should help most.

- Vanilla FasterRisk ($\mu = 0$) at $p = 211$
- Causal FasterRisk ($\mu > 0$) at $p = 211$

Primary metric: support stability across CV folds, not raw AUC (see §6.5).

**Experiment 3.2 — Prevalence-filtered set ($p = 17$, both solvers).** The full $2 \times 2$ matrix at matched $(n, p, k)$:

|  | FasterRisk | RiskSlim |
|---|---|---|
| Vanilla ($\mu = 0$) | beam-search baseline | exact-MIP baseline |
| Causal ($\mu > 0$) | the method | the method, exact |

The RiskSlim-vanilla cell anchors whether FasterRisk's beam search is close to optimal at this dimension (it should be). The RiskSlim-causal cell proves the prior's gain is not a beam-search artifact. Cross-solver agreement at both $\mu$ values is the strongest internal-validity evidence the contribution gets — the prior, not the search algorithm, is doing the work.

**Experiment 3.3 — Post-causal-selection scorecard ($p = 5$–$7$, RiskSlim certified-optimal).** Take the survivors from §7.3 (`pepq_Ala87Gly` + lineage indicators + any second-pass causal candidates), run RiskSlim to MIP optimality at $k = 3$, report a single certified-optimal integer scorecard. Not a comparison: an exhibit. The narrative claim is "causal pre-selection shrinks the post-selection problem enough to admit a global optimum; here is the resulting clinical-grade scorecard." Caveat: at $p = 5$, $k = 3$, FasterRisk also finds the optimum (the search space is small) — the value is the optimality certificate and the clean publication figure, not a head-to-head win for RiskSlim.

### 6.4 Baselines and ablations

- **Vanilla FasterRisk** ($\mu = 0$). Cross-track.
- **Vanilla RiskSlim** ($\mu = 0$). Cross-track wherever $p$ is MIP-tractable.
- **Hard pre-selection** ($\mu \to \infty$). Tests the §2.3 upper limit.
- **$L_1$ logistic regression.** Cross-track.
- **XGBoost** (capacity upper bound, not an interpretability competitor).
- **Uniform $\mathbf{q}$ ablation.** Same $\mu$, $q_j$ constant; tests whether the *structure* of the prior, not its strength alone, carries the signal.
- **Bootstrap-$L_1$-stability as $\mathbf{q}$ ablation.** Central ablation for the contribution: tests whether *causal* structure carries information beyond *predictive* structure. If the causal prior beats the bootstrap-$L_1$ prior at matched $\mu$, the causal selection is doing real work.
- **Modern causal feature selection.** One or two concrete competitors from 2023–2025 (causal-LASSO variants, double-machine-learning post-selection inference, doubly-robust feature screening), selected before paper draft.

### 6.5 Metrics and honest concerns

**Primary metrics.** Test AUC with confidence intervals; calibration (Brier and/or ECE); **support stability across CV folds** (Jaccard set-overlap averaged over folds); number of features in the score; runtime.

**Why support-stability is the headline on TB.** At $p \gg n$ with small effect sizes (Exp. 3.1), test AUC is a poor discriminator between methods — confidence intervals overlap. *Which features each method picks across folds* discriminates better. Vanilla at $p = 211$ is expected to pick different 5-feature supports across folds; the causal arm should pick stably overlapping supports including `pepq_Ala87Gly`. This is the figure that tells the story.

**Honest concerns.**

- $n = 164$ is small; all CV-derived metrics carry wide intervals. Report intervals, not point estimates.
- $q_j = 0$ for prevalence-filtered-out features encodes "no causal evidence available," not "known non-causal." State explicitly wherever the prior is described.
- Post-causal-selection RiskSlim (Exp. 3.3) is an exhibit, not a comparison. The value is the optimality certificate, not a head-to-head.
- Cross-solver agreement at $\mu = 0$ (Exp. 3.2, §6.1, §6.2) is a prerequisite, not a result. If FasterRisk and RiskSlim diverge at $\mu = 0$, debug before publishing $\mu > 0$ comparisons.

## 7. Application: TB-DLM resistance

### 7.1 Data and targets

$n = 164$ TB-MDR strains tested against delamanid; $p \approx 14$ mutations after prevalence filtering at $[5\%, 98\%]$; 3 lineage indicators with lineages 1 ($n = 3$) and 3 ($n = 4$) merged into reference; `type_beyond_MDR` (clinical preXDR/XDR vs. MDR) as an optional sensitivity covariate.

**The natural target is continuous.** $t = \log_2(\mathrm{MIC})$ on a 2-fold dilution series. The CMM prior-estimation stage operates directly on $t$: MIC is a sink node in the DAG with a linear-Gaussian mechanism, while the binary mutation nodes use the logistic mixture mechanism (FLXMRglm).

**Binarization is forced by FasterRisk, not by the problem.** FasterRisk requires $y \in \{-1, +1\}^n$, so we obtain it as $y = 2 \cdot \mathbb{1}[\mathrm{MIC} > \tau] - 1$. Three thresholds compared (median split, top quartile, agar-based `interp_dlm016`); binarization sensitivity is part of the evaluation.

This two-stage continuous-then-binary architecture is application-specific. Applications whose natural target is already binary (the §6.2 benchmarks) collapse this to one stage.

### 7.2 Prior source: CMM-logistic with stability selection

CMM (Mameche et al.) extends score-based Bayesian-network learning with per-node latent mixture components. The logistic extension (FLXMRglm, binomial link) handles binary mutation predictors directly. Continuous nodes (MIC) use the linear-Gaussian mechanism. Latent components proxy for unobserved patient-level confounders (treatment history, host factors).

**Defensibility for TB.** (i) Latent confounders are real and unobserved at this dataset's collection time. (ii) Features are predominantly binary with low-prevalence tails, ill-suited to PC/GES. (iii) $n = 164$ rules out heavier latent-variable structure-learners (LMB-CSEM, EM over latent subspaces). (iv) CMM handles a continuous target node natively, which matches MIC's data type. The argument is local to TB-shaped data; other applications use the cheaper sources in §3.1.

**Forbidden edges.** $\mathrm{MIC} \to \mathrm{mutation}$ (temporal/mechanistic); edges into lineage (lineage fixed at infection); edges into `type_beyond_MDR` (set independently of DLM). Optionally $\mathrm{lineage} \to \mathrm{MIC}$ forbidden when treating lineage strictly as a covariate. Under these constraints MIC is a DAG sink, so $\mathrm{MB}(\mathrm{MIC}) = \mathrm{Pa}(\mathrm{MIC})$ (Aliferis et al. 2010).

**Stability selection.** $B = 100$ runs at 80% subsampling. Per-run column filter at $\mathrm{mcc} \times k_{\max} = 24$ positives (below this, $k$-component logistic mixtures collapse on the binary mutation nodes). Eligibility-weighted per-edge frequencies. The 80% subsampling fraction trades the M-B familywise-error guarantee for finite-sample power at $n = 164$.

### 7.3 Findings

Four stability-selection ablations completed: baseline, +lineage, +lineage+type, +lineage with $\mathrm{type} \to \mathrm{MIC}$ forbidden. Decision rule for causal credibility: variant must be stable in both `+lineage` and the type-forbid spec.

- `pepq_Ala87Gly`: stable in both (0.51 in `+lineage`, 0.61 in the type-forbid spec). Only clean candidate.
- `mmpl5_Asp767Asn`: appears in the type-forbid spec only; identifiability artifact via lineage_2 collinearity ($\rho = 0.95$).
- `rv1979c_C-135G`: monotonically degrades across specs; not credible.

Single-strain-collection caveat: $n = 164$ from one geographic/clinical cohort. Replication on an independent cohort is required before any clinical claim about `pepq_Ala87Gly`.

## 8. Scope and limitations

- **Prior source selection is application-dependent.** No universal best causal source; choice depends on assumed structure (causal sufficiency or not) and data shape (binary/continuous, $n$, target type).
- **$\mu$ tuning is empirical.** No closed-form scaling against $n$.
- **CMM defensibility is local to TB-shaped data** (binary low-prevalence features, latent confounders, small $n$, continuous target). Other applications use simpler causal sources.
- **Faithfulness** is required wherever a causal graph is the prior source; failures show up as biased $\mathbf{q}$ and are characterized by the synthetic robustness analysis (§6.1).
- **Magnitude-vs-support tension at large $\mu$.** The bonus can drive features into the support with small $|w_j|$ that AuxiliaryLossRounding may eliminate when the StarRaySearch multiplier $m$ is too small. Empirically mitigated by FasterRisk's multiplier search; not a theoretical guarantee.
- **Binarization is a lossy compromise on TB.** Continuous MIC carries sub-threshold variation about treatment-outcome risk that the binary call discards. The two-stage architecture mitigates this at the prior-estimation stage but not at the classifier stage; threshold sensitivity is reported.
- **Independence of $z_j$ across $j$ ignores feature redundancy.** Under strong collinearity (TB's lineage_2 / `mmpl5_Asp767Asn` at $\rho = 0.95$), per-feature $q_j$ does not capture the "select one, not both" structure of the optimal support. The hard sparsity cap mitigates this only when $k$ is tight relative to the redundancy. Coupled priors (DPPs, Boltzmann machines on $\mathbf{z}$) would address this but destroy per-feature decomposability and CP1/CP3 ranking.

## 9. Methodological contributions

1. **Bayesian causal-prior penalty for sparse integer classification.** Bernoulli inclusion prior with sigmoid link yields a linear-in-$\mathbf{q}$ MAP bonus. Threshold-free one-parameter family with vanilla FasterRisk and hard pre-selection as the two limits.
2. **Decomposability-preserving integration into FasterRisk.** Linear separability of the bonus preserves SparseBeamLR and CollectSparseDiversePool per-feature decomposability; AuxiliaryLossRounding bound (Thm. 3.1) transfers without modification.
3. **Solver-agnostic MAP construction.** The same prior integrates into FasterRisk's beam search (CP1–CP6 in the implementation cell) and into RiskSlim's MIP formulation (a single linear coefficient on the inclusion indicator); dual-solver evaluation (§6) demonstrates the contribution is the prior, not the search algorithm.
4. **A general interface for causal evidence in sparse integer classifiers.** The method composes with any procedure outputting a causal $\mathbf{q} \in [0, 1]^p$ (§3.1). The prior-estimation target need not be binary; only the FasterRisk classifier stage requires binary input. The causal-only restriction is a feature of the contribution, not a limitation: predictive-only priors are excluded because they add no information beyond the loss.
5. **Logistic extension of CMM** for binary observed variables (FLXMRglm binomial link integrated into the mixture-EM structure-learning loop), as the TB-specific prior source and the only defensible causal source for TB-shaped data with a continuous MIC target.

# Implementation plan: FasterRisk modification

Reference for the FasterRisk code changes implementing the modified objective from cell 0, §2.1. The implementation is source-agnostic: it does not depend on how $\mathbf{q}$ is produced, only on its values.

**Notation (math ↔ code).** Following Liu et al. (2022):

| Math | Code | Meaning |
|---|---|---|
| $\mathbf{w} \in \mathbb{Z}^p$ | `self.betas` | coefficient vector |
| $w_0$ | `self.beta0` / intercept | intercept |
| $\mathbf{w}^+$ | rounded `betas` | integer-rounded coefficients |
| $L(\mathbf{w}, w_0, \mathcal{D})$ | `compute_logisticLoss_*(...)` | logistic loss |
| $k$ | `k` (or sparsity arg) | sparsity bound |
| $C$ | `original_lb` / `original_ub` | coefficient box bound |
| $\lambda_2$ | `self.lambda2` | $L_2$ ridge (default $10^{-8}$) |
| $m$ | StarRaySearch multiplier | multiplier $m > 0$ |
| $\mathbf{q} \in [0,1]^p$ | `self.freq` | causal-prior vector (new) |
| $\mu \geq 0$ | `self.mu` | prior strength (new) |

The numpy array `self.freq` has shape $(p,)$ indexed consistently with feature columns. The name `freq` reflects its origin as stability frequencies on a causal graph, but the implementation does not require this.

**Backward-compatibility guarantee.** At $\mu = 0$ or `freq = zeros(p)`, the modified code is numerically equivalent to vanilla FasterRisk.

## 1. FasterRisk in three stages

(Algorithm 1 in Liu et al. 2022.)

1. **SparseBeamLR** (Alg. 2): beam search for a sparse continuous coefficient vector $(\mathbf{w}^*, w_0^*)$ with $\|\mathbf{w}^*\|_0 \leq k$, $\|\mathbf{w}^*\|_\infty \leq C$. Calls `ExpandSuppBy1` (Alg. 4) per beam level. Selection by smallest logistic loss at fixed support size.
2. **CollectSparseDiversePool** (Alg. 5): given $(\mathbf{w}^*, w_0^*)$, build a Rashomon pool $\mathcal{P}$ of near-optimal alternatives by single-feature swaps. Accept a swap if its loss is within $(1 + \epsilon) L^*$.
3. **StarRaySearch** (Alg. 3) + **AuxiliaryLossRounding** (Alg. 6): for each pool member, search over multipliers $m > 0$ and round to integers $\mathbf{w}^+$. Multiplicative loss-error bound (Thm. 3.1).

The modification targets stages 1 and 2 (support selection). Stage 3 is untouched because it operates on fixed supports; the bonus is invariant under any operation preserving support.

## 2. Change-point inventory

### 2.1 Files

All under `external/fasterrisk/src/fasterrisk/`:

| File | Modification |
|---|---|
| `sparseBeamSearch.py` | 2 sites (CP1, CP2) |
| `sparseDiversePool.py` | 3 sites (CP3, CP4, CP5) |
| `fasterrisk.py` | constructor threading (CP6) |
| `wrapper.py` | constructor threading (CP6) |
| `base_model.py`, `rounding.py`, `utils.py` | no modification |

Pure Python. No Cython, no C++.

### 2.2 CP1: gradient ranking in beam-search expansion

**File:** `sparseBeamSearch.py`, inside `expand_parent_i_support_via_OMP_by_1`, around lines 42-46.

**Original:**
```python
grad_on_non_support = self.yXT[non_support].dot(np.reciprocal(1 + self.ExpyXB_arr_parent[i]))
abs_grad_on_non_support = np.abs(grad_on_non_support)
num_new_js = min(child_size, len(non_support))
new_js = non_support[np.argsort(-abs_grad_on_non_support)][:num_new_js]
```

Picks the top `child_size` non-support features to expand into. Ranking is by $|\nabla_j L|$ at coefficient zero (first-order Taylor proxy for loss reduction).

**Modified:**
```python
grad_on_non_support = self.yXT[non_support].dot(np.reciprocal(1 + self.ExpyXB_arr_parent[i]))
abs_grad_on_non_support = np.abs(grad_on_non_support) + self.mu * self.freq[non_support]
num_new_js = min(child_size, len(non_support))
new_js = non_support[np.argsort(-abs_grad_on_non_support)][:num_new_js]
```

**Justification.** The first-order marginal change in $L_{\text{mod}}$ when adding feature $j$ is approximately $-|\nabla_j L| - \mu \cdot q_j$. Ranking by $|\nabla L| + \mu \cdot \mathbf{q}$ in descending order ranks by first-order modified-objective reduction.

**Limit checks.** $\mu = 0$: identical to vanilla. $\mu \to \infty$: pure top-$k$ by $\mathbf{q}$.

Strictly, the optimal first-order reduction of $L_{\mathrm{mod}}$ on adding feature $j$ is $|\nabla_j L|^2/(2H_{jj}) + \mu q_j$; under FasterRisk's standardization $H_{jj}$ is approximately constant across $j$, so ranking by $|\nabla_j L| + \mu q_j$ is rank-equivalent to ranking by the optimal reduction up to the same approximation vanilla FasterRisk uses. CP2 corrects to exact $L_{\mathrm{mod}}$ at the beam-survivor step.


### 2.3 CP2: child-loss in beam-search keep/discard

**File:** `sparseBeamSearch.py`, line 82.

**Original:**
```python
self.loss_arr_child[child_id] = compute_logisticLoss_from_ExpyXB(self.ExpyXB_arr_child[child_id])
```

Stored child loss is used at line 100 in `np.argsort(self.loss_arr_child)` to select the top `parent_size` children for the next beam level. This is where the actual beam selection happens.

**Modified:**
```python
support_idx = get_support_indices(self.betas_arr_child[child_id])
self.loss_arr_child[child_id] = (
    compute_logisticLoss_from_ExpyXB(self.ExpyXB_arr_child[child_id])
    - self.mu * self.freq[support_idx].sum()
)
```

**Justification.** Without this, CP1 biases candidate *generation* but the beam *survivor* selection still uses raw $L$, inconsistent with the modified objective.

**$L_2$ ridge asymmetry note.** Vanilla FasterRisk stores raw $L$ here (no $\lambda_2 \|\mathbf{w}\|^2$), while the diverse pool (CP4) stores $L + \lambda_2 \|\mathbf{w}\|^2$. We preserve this vanilla quirk: subtract bonus from raw $L$ in CP2, from $L + \lambda_2 \|\mathbf{w}\|^2$ in CP4. Required for numerical equivalence to vanilla at $\mu = 0$.

### 2.4 CP3: gradient ranking in diverse-pool swap

**File:** `sparseDiversePool.py`, inside `get_sparseDiversePool`, around lines 84-88.

**Original:**
```python
grad_on_availableIndices = -self.yXT[availableIndices].dot(np.reciprocal(1 + sparseDiversePool_ExpyXB[sparseDiversePool_start]))
abs_grad_on_availableIndices = np.abs(grad_on_availableIndices)
new_js = availableIndices[np.argsort(-abs_grad_on_availableIndices)[:max_num_new_js]]
```

**Modified:**
```python
grad_on_availableIndices = -self.yXT[availableIndices].dot(np.reciprocal(1 + sparseDiversePool_ExpyXB[sparseDiversePool_start]))
abs_grad_on_availableIndices = np.abs(grad_on_availableIndices) + self.mu * self.freq[availableIndices]
new_js = availableIndices[np.argsort(-abs_grad_on_availableIndices)[:max_num_new_js]]
```

Structurally identical to CP1. Swap-candidate selection biased toward features with high causal evidence.

### 2.5 CP4: pool-loss for final argsort

**File:** `sparseDiversePool.py`, lines 66 (seed-solution loss) and 102 (accepted-swap loss).

**Modified line 66:**
```python
sparseDiversePool_loss[-1] = (
    compute_logisticLoss_from_ExpyXB(self.ExpyXB)
    + self.lambda2 * self.betas[nonzero_indices].dot(self.betas[nonzero_indices])
    - self.mu * self.freq[nonzero_indices].sum()
)
```

**Modified line 102:**
```python
swap_support = np.setdiff1d(np.append(nonzero_indices, new_j), [old_j])
sparseDiversePool_loss[sparseDiversePool_index] = (
    compute_logisticLoss_from_ExpyXB(sparseDiversePool_ExpyXB[sparseDiversePool_index])
    + self.lambda2 * (betas_no_old_j_squareSum + sparseDiversePool_betas[sparseDiversePool_index, new_j] ** 2)
    - self.mu * self.freq[swap_support].sum()
)
```

**Justification.** At line 104, `np.argsort(sparseDiversePool_loss)[:totalNum_in_diverseSet][:select_top_m]` selects the final pool. Subtracting the bonus makes the pool surface near-optimal solutions under the modified objective.

### 2.6 CP5 (deliberate non-modification): gap-tolerance check stays on raw loss

**File:** `sparseDiversePool.py`, line 95.

**Why this is not modified.** The gap check is multiplicative-relative: $(L_{\text{swap}} - L_{\text{ref}}) / L_{\text{ref}} < \text{gap\_tolerance}$. At large $\mu$, $L_{\text{mod}}$ can go negative ($\mu \sum_j q_j$ exceeds $L + \lambda_2 \|\mathbf{w}\|^2$); dividing by a negative reference flips the inequality and the gap check accepts arbitrarily bad swaps. Keep the gap comparison on raw $L + \lambda_2 \|\mathbf{w}\|^2$ (always nonnegative).

**Semantic consequence.** Pool members are "near-optimal in raw $L + \lambda_2 \|\mathbf{w}\|^2$, ranked by modified objective." Acknowledged drawback: at large $\mu$, a feature with high $q_j$ but no predictive evidence may be ranked highly by CP3 but its swap fails the raw-loss gap check, so it never enters the pool. The bonus is effectively muted for pool *membership* at large $\mu$, even though it still determines pool *ordering*. Alternatives considered:
- Absolute gap rather than relative: avoids the sign issue but loses scale invariance.
- Gap on $|L_{\text{ref}}|$: rescues sign at the cost of meaning.
- Gap on $L + \lambda_2 \|\mathbf{w}\|^2$ explicitly: current choice. Cleanest.

**Implementation pattern.** Store the modified-objective values in `sparseDiversePool_loss`; reconstruct raw loss at the gap-check site:

```python
loss_ref_raw = sparseDiversePool_loss[-1] + self.mu * self.freq[nonzero_indices].sum()
loss_swap_raw = (
    compute_logisticLoss_from_ExpyXB(sparseDiversePool_ExpyXB[sparseDiversePool_index])
    + self.lambda2 * (betas_no_old_j_squareSum + sparseDiversePool_betas[sparseDiversePool_index, new_j] ** 2)
)
if (loss_swap_raw - loss_ref_raw) / loss_ref_raw < gap_tolerance:
    # ... accept swap
```

### 2.7 CP6: parameter threading

Two new constructor parameters, `mu: float = 0.0` (math: $\mu$) and `freq: np.ndarray = None` (math: $\mathbf{q}$), threaded from `wrapper.py` and `fasterrisk.py` down to `sparseLogRegModel` and `sparseDiversePoolLogRegModel`.

**Validation in each model constructor:**
```python
self.mu = float(mu)
self.freq = np.asarray(freq) if freq is not None else np.zeros(X.shape[1])
assert self.freq.shape == (X.shape[1],), \
    f"freq must have shape ({X.shape[1]},), got {self.freq.shape}"
assert self.mu >= 0, "mu must be nonnegative"
```

Default `mu=0.0, freq=zeros(p)` makes the modification a no-op. With either default, CP1-CP4 reduce to vanilla expressions; numerical equivalence is the backward-compatibility test.

### 2.8 What does NOT need modification

| Component | Why no modification |
|---|---|
| `base_model.py`: `finetune_on_current_support` | Optimizes coefficients on a fixed support. Bonus is constant in the coefficients. |
| `rounding.py`: `AuxiliaryLossRounding` (Alg. 6) | Rounds magnitudes; never alters support. |
| `fasterrisk.py`: `StarRaySearch` (Alg. 3) | Multiplier $m$ search scales coefficients uniformly; support invariant. |
| Group-sparsity subclasses | Only override `getAvailableIndices_*`; our bonus is added *after* this filter. |
| `utils.py` | We don't redefine helpers; bonus is added at call sites. |

The integer-rounding bound transfers exactly because the bonus is invariant under support-preserving operations (§3 below).

## 3. Integer-rounding bound transfer

Liu et al. 2022 (Thm. 3.1): for the continuous sparse solution $\mathbf{w}$ and the integer-rounded $\mathbf{w}^+$,

$$
L(\mathbf{w}^+, w_0^+, \mathcal{D}) - L(\mathbf{w}, w_0, \mathcal{D}) \;\leq\; \sqrt{\, n \sum_{i=1}^n \sum_{j=1}^p \ell_i^2 \, x_{ij}^2 \, u_j (1 - u_j) \,},
$$

with $u_j = w_j - \lfloor w_j \rfloor$ and $\ell_i = 1/(1 + \exp(y_i \mathbf{x}_i^\top \boldsymbol{\gamma}_i))$ the local Lipschitz constant ($\gamma_{ij} = \lfloor w_j \rfloor$ if $y_i x_{ij} > 0$, else $\lceil w_j \rceil$).

For the modified objective:

$$
\begin{aligned}
L_{\text{mod}}(\mathbf{w}^+, w_0^+) - L_{\text{mod}}(\mathbf{w}, w_0)
&= \big[\, L(\mathbf{w}^+, w_0^+, \mathcal{D}) - L(\mathbf{w}, w_0, \mathcal{D}) \,\big] \\
&\quad + \lambda_2 \big[\, \|\mathbf{w}^+\|^2 - \|\mathbf{w}\|^2 \,\big] \\
&\quad - \mu \sum_{j=1}^p q_j \big[\, \mathbb{1}[w_j^+ \neq 0] - \mathbb{1}[w_j \neq 0] \,\big].
\end{aligned}
$$

Third bracket: identically zero (support invariant under rounding). Second bracket: small magnitude-change term vanilla FasterRisk also incurs, default $\lambda_2 = 10^{-8}$ negligible. First bracket: bounded by Thm. 3.1.

The bonus is *structurally* invariant under rounding, not just bounded under rounding. Caveat: support preservation assumes the StarRaySearch multiplier $m$ is large enough that no nonzero $|m \cdot w_j|$ rounds to zero. The bonus can drive features into the support with small $|w_j|$ (the magnitude-vs-support tension, §5.3); empirically StarRaySearch picks $m$ large enough on the regimes Liu et al. tested. Not a theoretical guarantee at extreme $\mu$.

## 4. Sanity checks

Run after implementation, before any new-source or benchmark work.

1. **$\mu = 0$ numerical equivalence.** Run existing FasterRisk tests with modified code. Outputs must match vanilla within numerical tolerance. Most important check.
2. **$\mathbf{q} = \mathbf{0}$ numerical equivalence.** Pass $\mu > 0$ but `freq = zeros(p)`. Outputs must match vanilla.
3. **$\mu \to \infty$ picks top-$k$ by $\mathbf{q}$.** Controlled instance: small synthetic where loss is approximately constant across supports (random labels, balanced features). Set $\mu = 10^6$; verify selected support equals top-$k$ features by $\mathbf{q}$.
4. **Monotonicity probe.** Sweep $\mu \in \{0, 0.01 \cdot s, 0.1 \cdot s, s, 10 s, \infty\}$ with $s = \mathrm{median}_j |\nabla_j L|$ on a synthetic with planted causal support. Plot: fraction of planted causal features selected vs. $\mu$. Expected: monotonic increase (modulo beam heuristics).
5. **Rashomon pool sanity.** At $\mu = 0$, pool $\mathcal{P}$ equals vanilla pool. At $\mu > 0$, pool members' raw losses are within `gap_tolerance` of the optimum.

## 5. Numerical and structural subtleties

### 5.1 The $\mu \to \infty$ limit is not literally "top-$k$ by $\mathbf{q}$"

`forbidden_support` (sparseBeamSearch.py line 77) deduplicates supports across the beam. As parents converge toward $\mathbf{q}$-top supports, dedup blocks duplicate proposals; the beam may end with fewer than `parent_size` surviving members. The empirical behavior is "beam concentrates on the highest-$\mathbf{q}$ supports"; the formal statement is "in the absence of beam heuristics, $k$-optimum is top-$k$ by $\mathbf{q}$."

### 5.2 Scale of $\mu$ is data-dependent

$|\nabla_j L|$ depends on $n$ and feature scale; $q_j \in [0, 1]$ does not. The sum $|\nabla L| + \mu \cdot \mathbf{q}$ in CP1 mixes scales. Practical convention: report and sweep $\mu$ in units of $\mathrm{median}_j |\nabla_j L|$ at $\mathbf{w} = \mathbf{0}$. Sweep $\{0, 0.01, 0.1, 1, 10, \infty\} \cdot \mathrm{median}|\nabla L|$, not raw values.

### 5.3 Magnitude vs support tension at large $\mu$

When the bonus heavily favors a feature, the OMP inner loop (Eq. 7 in Liu et al.) finds its optimal coefficient even if small. Low-utility features can sit in the support with near-zero $|w_j|$, kept by the bonus. AuxiliaryLossRounding combined with a small StarRaySearch multiplier $m$ may then eliminate them, undoing the bonus. Intended behavior under "causal override" framing, but caps the useful range of $\mu$ on small-effect features. Characterize empirically in stage 4.

### 5.4 Feature indexing

- $X$ is $n \times p$. `non_support`, `support` arrays in `sparseBeamSearch.py` and `sparseDiversePool.py` index the $p$ feature columns ($j = 1, \ldots, p$).
- Intercept $w_0$ stored separately as `self.beta0`; not indexed in `non_support` / `support`.
- `freq` must be length $p$, indexed consistently with $X$ columns.
- FasterRisk standardizes features (`X_norm`, `scaled_feature_indices`) but does not permute them. Verify once by reading `base_model.py` before committing.

### 5.5 $L_2$ ridge handling is asymmetric (vanilla quirk preserved)

`sparseBeamSearch.py` stores raw $L$ in `loss_arr_child` (no $\lambda_2 \|\mathbf{w}\|^2$). `sparseDiversePool.py` stores $L + \lambda_2 \|\mathbf{w}\|^2$. Vanilla FasterRisk's choice, not ours. Preserved so $\mu = 0$ matches vanilla bit-by-bit on the standard arithmetic path.

## 6. Pre-flight reads (do these before writing code, ~30 min)

1. **`base_model.py` end-to-end.** Confirm no selection-critical paths outside the two files being modified. Low probability of a surprise, worth verifying.
2. **Feature ordering.** Confirm features are not permuted by standardization (i.e., `self.betas[j]` corresponds to `X[:, j]`). If permuted, threading `freq` requires the same permutation.
3. **`forbidden_support` dedup.** Uses string keys of support sets. Confirm it does not depend on loss values (should be fine — runs before loss computation — but verify).

## 7. Expected effort

| Step | Estimate |
|---|---|
| Pre-flight reads | 30 min |
| Implement CP1, CP2 | 1 hour |
| Implement CP3, CP4, CP5 | 2 hours |
| Implement CP6 (threading) | 1 hour |
| Sanity check 1 ($\mu = 0$ equivalence) | 1 hour |
| Debug if check 1 fails | 0-4 hours |
| Sanity checks 2-5 | 2 hours |
| Buffer for numerical-tolerance surprises | 6-8 hours |
| **Total** | **~1 week realistic, 2-3 days optimistic** |

The original "1-2 days" estimate ignored that numerical-equivalence-at-$\mu=0$ debugging can absorb a day on its own. Floating-point reordering from the added arithmetic in CP1/CP2/CP4 is the likely failure mode.

The choice is not a smoothness preference. It is the unique link making the log-prior linear in $q_j$.

Start with Bernoulli on $z_j = \mathbb{1}[w_j \neq 0]$:

$$\log p(z_j) = z_j \log \pi_j + (1 - z_j) \log(1 - \pi_j) = z_j \cdot \mathrm{logit}(\pi_j) + \log(1 - \pi_j).$$

The first term is the only one that depends on $z_j$; the second is a $j$-only constant in the MAP. So the support-dependent part of the log-prior is

$$\sum_j z_j \cdot \mathrm{logit}(\pi_j).$$

We want this to equal $\mu \sum_j q_j z_j$ (linear-in-$\mathbf{q}$ bonus). Matching coefficients:

$$\mathrm{logit}(\pi_j) = \mu q_j \iff \pi_j = \sigma(\mu q_j).$$

This is the only $\pi_j$ that delivers a linear-in-$q$ MAP bonus through the Bernoulli inclusion prior. Any other link (e.g., $\pi_j = q_j$ directly, or $\pi_j = q_j^\mu$) produces a non-linear log-prior with no closed-form decomposability across features — i.e., destroys the CP1/CP3 per-feature ranking. So calling $\sigma(\mu q_j)$ "natural" understates it; it's forced by two design choices made earlier: Bernoulli prior + linear-in-$q$ bonus.



Vanilla FasterRisk reasoning. At parent $(\mathbf{w}{\text{parent}}, w_0)$, consider adding feature $j \notin \mathrm{supp}(\mathbf{w}{\text{parent}})$ with coefficient $\delta$:

$$L(\mathbf{w}{\text{parent}} + \delta \mathbf{e}j) - L(\mathbf{w}{\text{parent}}) = \delta \cdot \nabla_j L + \tfrac{1}{2} \delta^2 H{jj} + O(\delta^3).$$

Minimizing over $\delta$ gives optimal step $\delta^* \approx -\nabla_j L / H_{jj}$ and loss reduction $\approx (\nabla_j L)^2 / (2 H_{jj})$. After standardization $H_{jj}$ is roughly constant across $j$, so the loss reduction is monotonic in $|\nabla_j L|^2$, equivalently in $|\nabla_j L|$. That's the vanilla rank: top-child_size by $|\nabla_j L|$.

With the bonus. $L_{\text{mod}}$ adds the support-indicator term $-\mu q_j z_j$. When $j$ moves from $z_j = 0$ to $z_j = 1$, the bonus contributes a discrete drop of $-\mu q_j$, independent of the coefficient magnitude $\delta$. So:

$$L_{\text{mod}}(\mathbf{w}{\text{parent}} + \delta \mathbf{e}j) - L{\text{mod}}(\mathbf{w}{\text{parent}}) = \underbrace{\delta \nabla_j L + \tfrac{1}{2}\delta^2 H_{jj}}{\text{loss part}} ;-; \underbrace{\mu q_j}{\text{support flip}}.$$

At optimal $\delta^*$ the loss part is approximately $-|\nabla_j L|^2 / (2 H_{jj})$. The total optimal-step reduction is then:

$$|\Delta L_{\text{mod}}^*| \approx \frac{|\nabla_j L|^2}{2 H_{jj}} + \mu q_j.$$

Ranking by this quantity would be strictly correct under the quadratic-Taylor + standardized-Hessian assumption. The code uses the simpler proxy

$$\text{score}_j = |\nabla_j L| + \mu q_j,$$

which is rank-equivalent to $|\nabla_j L|^2 + 2 H_{jj} \mu q_j$ only under additional approximations. The simpler proxy is what vanilla FasterRisk already uses (ranking by $|\nabla_j L|$ rather than $|\nabla_j L|^2$); CP1 just adds the discrete-flip cost on the same scale.